- Download the historical data with yfinance (choose one asset that you like, e.g. S&P500, WIG, etc.)
- The x_n is data for day n (Open, Close, Volume, Etc.)
y_n is 1 if the price increased on day (n+1) and 0 otherwise
- The aim is to predict if the price increases on the next day
- Choose a model (eg. logistic regression with regularisation) train the data, find the best metaparametrs.
- Note that the x_n data might need a transformation. Eg. log volume seems to be a better idea than volume.
- Observe the performance on a test dataset
- Submit everything as a notebook, in a clear and concise form. (Conclusions!)

In [ ]:
# standard 
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import scipy.stats as stats

# models 
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report, confusion_matrix

# data
import yfinance as yf


---

## Solution

### Task

Create a model that decides **whether to buy today and sell the next day** (using **only the opening price** as input).

### Solution

#### Goal

I wanted a model that predicts the value on day \(i\) based on the last \(n\) days. Through some experimentation, \(n=3\) gave the best results, so all testing was conducted using a 3-day input window.

#### Methodology

1. **Dataset**
   - I used the *"PKO.WA"* ticker (Polish stock PKO Bank).
2. **Splitting Data**
   - I took 600 days of historical data for training, setting up features with length \(n\) and then the target for day \(i\).
   - These training “probes” do not overlap (so each chunk is disjoint). For the test set, however, the data **does** overlap, but is taken from later dates not used in training.
   - **Note**: I selected a specific time interval for training, so there might be bias due to that choice. In a real scenario, a rolling  window approach might be more robust.
3. **Model Training**
   - I fit a **Logistic Regression** model. I chose logistic regression to classify each day’s movement as profit/loss (binary), which seems suitable for a yes/no decision.

#### Results & Discussion

- **Results**:

  - The F1-score is fairly low for the negative (0) class (0.22) and moderate for the positive (1) class (0.69). However, we see that recall for class 1 (the “buy” or “profit” class) is 0.88, meaning the model predicts “positive movement” quite often, capturing many actual upward moves.
  - A simple profit simulation, taking a long position on days classified as “buy,” yields a net cumulative gain over time (as shown in the line plot).

- **Discussion**:

  - The approach uses **only the opening price** and checks whether tomorrow’s open is higher than today’s. We could improve this by incorporating *close prices*, daily volatility, or additional technical indicators.
  - The data selection and window approach can be refined—currently, the 600-day chunk may not be representative of all market conditions. Possibly **rolling** or **sliding** windows might produce more robust and up-to-date models.
  - Selecting 600 days from a particular market regime might bias the model. Real markets can shift behavior (bullish vs. bearish periods). Ensuring data covers multiple phases (and then properly validating) is important.
  - The visualization is limited, but we do observe the model’s *cumulative profit curve* trending upward.

#####

##

In [ ]:
wig20 = yf.Ticker("PKO.WA")
hist = wig20.history(period="3y")


In [ ]:
def build_day_window_df(series: pd.Series, n_days: int = 5, amount: int = 100) -> pd.DataFrame:
    X_rows = []
    for i in range(0, len(series) - n_days, n_days + 2):
        window = series.iloc[i:i+n_days].tolist()
        next_day = series.iloc[i+n_days]
        target = int(next_day > window[-1])  # 1 jeśli wzrost, inaczej 0
        X_rows.append(window + [target])

    # Nazwy kolumn: Day 1, Day 2, ..., Day n, Target
    col_names = [f'Day {i+1}' for i in range(n_days)] + ['Target']
    return pd.DataFrame(X_rows, columns=col_names)

In [ ]:
# parameters
n = 3 # parameters amount 
k = 5 # probe amount 

# help functions 
def data_transform(DataFrame, idx):
    '''Function transforms data into data which can be calculated through model'''
    # single point
    if type(idx) is not list:
        rtn_val = DataFrame.iloc[idx-n:idx].values.reshape(-1, n)
        rtn_val = pd.DataFrame(rtn_val, columns=[f'Day {i+1}' for i in range(n)])
    # multiple points 
    else:
        row_list = list()
        # calculatting each probe
        for id in idx:
            row = DataFrame.iloc[id-n:id].values.reshape(-1, n)
            row = pd.DataFrame(row, columns=[f'Day {i+1}' for i in range(n)])
            row_list.append(row)
        # creating DataFrame 
        rtn_val = pd.concat(row_list)
    return rtn_val

# ----------------Creating sets------------------- #

# creating wanted df with feature we are looking for 
df = hist.assign(Profit=hist['Open'])[['Profit']]
# creating training set 
train_set = df.iloc[:100*(k+1)]['Profit'].values.reshape(-1, n+1)
train_set = pd.DataFrame(train_set, columns=[f'Day {i+1}' for i in range(n+1)])
# creating value prediction positive / negative 
train_set[f'Day {n+1}'] = (train_set[f'Day {n+1}']-train_set[f'Day {n}'] > 0).astype(int)
X_train, y_train = train_set[[f'Day {i+1}' for i in range(n)]], train_set[f'Day {n+1}']

# creating testing set 
test_set = df.iloc[100*(k+1) + 15:-10]['Profit']
idx = list(range(3 + n, 50 + n))
X_test = data_transform(test_set, idx)
y_test = ((test_set.values[idx] - test_set.values[np.array(idx) - 1]) > 0).astype(int)
# funkcja która wywołuje 5 poprzednich indeksów dla elemntu i robie z tego DataFrame'a 

# ----------------Training model------------------- #

# training model
model = LogisticRegression()
model.fit(X_train, y_train)
# scoring model
y_pred = model.predict(X_test)
# TODO make some visualization for reports 
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

# ----------------Predictions ------------------ #

# test function based on real profit 
def profit_test(df, model, start_day, end_day):
    profits = list()
    for id in range(start_day, end_day):
        # TODO predicting value
        X = data_transform(df, id)
        y = model.predict(X)
        # TODO calculting profit 
        if y == 1:
            prof = df['Profit'].iloc[id] - df['Profit'].iloc[id-1]
            profits.append(prof)
        else:
            profits.append(0)
    # TODO adding profit 
    profits = np.array(profits).cumsum()
    return profits 
    # returning values 
values = profit_test(df, model, 620, 700)

plt.plot(range(700-620), values)
# testing ~ standard
#  testing (via function that starts with money, and play some amount of rounds)